# 1.12 更新一次有什麼效果？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**負梯度配合減號，參數會變大還是變小？

要降低 loss，沿梯度反方向走一小步。這個一變量實驗只做一次更新，讓公式與程式對上；不跑模型訓練。

**把直覺寫成數學：**$w_{new}=w-\eta\nabla L(w)$；learning rate $\eta$ 決定走多遠。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
w = torch.tensor(1.0, requires_grad=True)
before = (w - 3).square()
before.backward()
with torch.no_grad():
    w -= 0.1 * w.grad
after = (w - 3).square()
print(float(before.detach()), float(after.detach()), w.detach())
assert after < before

**如何讀結果：**no_grad 是因為更新不是本次要微分的 forward；loss 降低不是對所有 learning rate 的保證。

**只改一件事：**只把 learning rate 改成 2，解釋可能跳過谷底。
